# 00 — Data Validation

Self-contained notebook: no `%run` dependency on project scripts.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import yaml

# In Colab, change only this path after mounting Google Drive.
PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "src" else Path(".").resolve()
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
with CONFIG_PATH.open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

def project_path(key):
    return PROJECT_ROOT / CONFIG["paths"][key]

OUTPUT = project_path("output")
PROCESSED = project_path("processed")
for folder in [OUTPUT / "reports", OUTPUT / "figures", OUTPUT / "models", OUTPUT / "labeling", OUTPUT / "logs", PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
SHOP_COLUMNS = ["Shop_ID", "Shop_Name", "Shop_type", "Years_Active", "Is_Online_Now", "Total_Products", "Follower_Count_k", "Rating_Average", "Total_Ratings_k", "Chat_Response_Rate_pct", "Has_Voucher", "Target_Label", "Time_Collected"]
PRODUCT_COLUMNS = ["Shop_ID", "product_id", "product_name", "product_details", "description_text", "Review_stars"]
REVIEW_COLUMNS = ["Shop_ID", "product_id", "review_id", "user_name", "rating", "review_time", "review_text", "has_image"]

def load_json_folder(folder, pattern, columns):
    records = []
    for path in sorted(folder.glob(pattern)):
        with path.open(encoding="utf-8") as handle:
            payload = json.load(handle)
        if not isinstance(payload, list):
            raise ValueError(f"{path}: top-level JSON must be a list")
        records.extend(payload)
    return pd.DataFrame(records) if records else pd.DataFrame(columns=columns)

shops = pd.read_excel(project_path("shops"), dtype={"Shop_ID": str})
products = load_json_folder(project_path("products"), "shop_*_products.json", PRODUCT_COLUMNS)
reviews = load_json_folder(project_path("reviews"), "shop_*_reviews.json", REVIEW_COLUMNS)
print({"shops": len(shops), "products": len(products), "reviews": len(reviews)})

In [ ]:
checks = []
def add_check(name, invalid, description):
    count = int(invalid.sum()) if isinstance(invalid, pd.Series) else len(invalid)
    checks.append({"check_name": name, "status": "PASS" if count == 0 else "FAIL", "error_count": count, "description": description})

for name, frame, required in [("shops", shops, SHOP_COLUMNS), ("products", products, PRODUCT_COLUMNS), ("reviews", reviews, REVIEW_COLUMNS)]:
    add_check(f"{name}_required_columns", set(required) - set(frame.columns), "Required schema")
    if frame.empty:
        checks.append({"check_name": f"{name}_has_rows", "status": "WARNING", "error_count": 0, "description": "No real records supplied"})

if all(set(required) <= set(frame.columns) for frame, required in [(shops, SHOP_COLUMNS), (products, PRODUCT_COLUMNS), (reviews, REVIEW_COLUMNS)]):
    add_check("unique_Shop_ID", shops["Shop_ID"].duplicated(), "Shop_ID unique")
    add_check("unique_product_id", products["product_id"].duplicated(), "product_id unique")
    add_check("unique_review_id", reviews["review_id"].duplicated(), "review_id unique")
    add_check("shop_rating_range", ~pd.to_numeric(shops["Rating_Average"], errors="coerce").between(0, 5), "Rating_Average in [0,5]")
    add_check("response_rate_range", ~pd.to_numeric(shops["Chat_Response_Rate_pct"], errors="coerce").between(0, 100), "Response rate in [0,100]")
    add_check("product_rating_range", ~pd.to_numeric(products["Review_stars"], errors="coerce").between(0, 5), "Review_stars in [0,5]")
    add_check("review_rating_range", ~pd.to_numeric(reviews["rating"], errors="coerce").between(1, 5), "rating in [1,5]")
    add_check("review_has_image", ~reviews["has_image"].isin([0, 1]), "has_image binary")
    add_check("product_shop_fk", set(products["Shop_ID"]) - set(shops["Shop_ID"]), "Product Shop FK")
    add_check("review_shop_fk", set(reviews["Shop_ID"]) - set(shops["Shop_ID"]), "Review Shop FK")
    add_check("review_product_fk", set(reviews["product_id"]) - set(products["product_id"]), "Review Product FK")
    product_shop = products.set_index("product_id")["Shop_ID"].to_dict()
    mismatch = reviews.apply(lambda row: product_shop.get(row["product_id"]) not in (None, row["Shop_ID"]), axis=1)
    add_check("review_product_shop_match", mismatch, "Review.Shop_ID matches Product.Shop_ID")

validation_report = pd.DataFrame(checks)
validation_report.to_csv(OUTPUT / "reports" / "data_validation_report.csv", index=False)
display(validation_report)
failed = validation_report.query("status == 'FAIL'")
if not failed.empty:
    raise ValueError("Critical validation failed: " + ", ".join(failed["check_name"]))